In [1]:
import sys
from pathlib import Path
import numpy as np
import torch
import pandas as pd
from chronos import Chronos2Pipeline

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from _pool_common import (
    load_pool_data,
    backtest_21d_rolling,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_CONTEXT_CHRONOS,
    ARTIFACTS_DIR,
    TICKERS,
)

MODEL_ID = "amazon/chronos-2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

c:\capstone_project_unfc\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(f"Loading {MODEL_ID} on {DEVICE}...")
pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.float32,
)

def get_forecast_chronos_price(context: pd.Series, horizon: int):
    """Chronos predicts price directly (no returns). Return list of `horizon` price point forecasts (0.5 quantile)."""
    try:
        context = context.astype(float)
        idx = pd.date_range(context.index.min(), context.index.max(), freq="B")
        context_regular = context.reindex(idx).ffill().bfill().dropna()
        if len(context_regular) < 3:
            return []
        target_values = context_regular.values.flatten()
        timestamps = context_regular.index
        input_df = pd.DataFrame({
            "item_id": ["x"] * len(target_values),
            "timestamp": timestamps,
            "target": target_values,
        })
        forecast_df = pipeline.predict_df(
            input_df,
            prediction_length=horizon,
            quantile_levels=[0.5],
            id_column="item_id",
            timestamp_column="timestamp",
            target="target",
            validate_inputs=True,
        )
        qcol = "0.5" if "0.5" in forecast_df.columns else "predictions"
        if qcol not in forecast_df.columns:
            qcol = forecast_df.columns[-1]
        q = forecast_df[qcol]
        vals = np.asarray(q).ravel()
        return [float(vals[i]) for i in range(min(horizon, len(vals)))]
    except Exception as e:
        print(f"Chronos forecast error: {e}")
        return []

Loading amazon/chronos-2 on cpu...


`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


In [3]:
stacked = load_pool_data()
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL     1256
AMZN     1256
GOOGL    1256
JNJ      1256
JPM      1256
MSFT     1256
NVDA     1256
SPY      1256
WMT      1256
XOM      1256
dtype: int64


,timestamp,symbol,close
0,2021-03-01,AAPL,127.790001
1,2021-03-02,AAPL,125.120003
2,2021-03-03,AAPL,122.059998
3,2021-03-04,AAPL,120.129997
4,2021-03-05,AAPL,121.419998
5,2021-03-08,AAPL,116.360001
6,2021-03-09,AAPL,121.089996
7,2021-03-10,AAPL,119.980003
8,2021-03-11,AAPL,121.959999
9,2021-03-12,AAPL,121.029999


In [4]:
# 21-day-ahead direct price forecast; 60-day test window, rolling by 7 days
model_name = "chronos_price"
all_preds = []
for sym in TICKERS:
    grp = stacked[stacked["symbol"] == sym]
    if grp.empty:
        continue
    prices = grp.set_index("timestamp")["close"].astype(float).dropna().sort_index()
    if len(prices) < TEST_SIZE + MIN_CONTEXT_CHRONOS:
        continue
    pred = backtest_21d_rolling(
        prices, TEST_SIZE, FORECAST_HORIZON, ROLLING_STEP,
        MIN_CONTEXT_CHRONOS,
        get_forecast_fn=get_forecast_chronos_price,
    )
    if pred.empty:
        continue
    pred["symbol"] = sym
    all_preds.append(pred)

pred_chronos = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"])
print(pred_chronos.groupby("symbol").size() if not pred_chronos.empty else "No predictions.")
pred_chronos.head()

c:\capstone_project_unfc\env\Lib\site-packages\chronos\chronos2\dataset.py:89: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  task_target = torch.from_numpy(task_target)


symbol
AAPL     126
AMZN     126
GOOGL    126
JNJ      126
JPM      126
MSFT     126
NVDA     126
SPY      126
WMT      126
XOM      126
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-02,286.190002,283.210602,0,AAPL
1,2025-12-03,284.149994,283.584778,0,AAPL
2,2025-12-04,280.700012,283.630371,0,AAPL
3,2025-12-05,278.779999,283.336975,0,AAPL
4,2025-12-08,277.890015,283.737427,0,AAPL


In [5]:
metrics_rows = []
for sym in pred_chronos["symbol"].unique():
    sub = pred_chronos[pred_chronos["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_chronos)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_chronos_pool_price.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_chronos_pool_price.parquet")

            model   symbol        MAE       RMSE    MAPE_%
0   chronos_price     AAPL  11.831006  13.642819  4.493258
1   chronos_price     MSFT  19.927920  23.966249  4.640731
2   chronos_price    GOOGL  11.495952  13.026068  3.565528
3   chronos_price     AMZN  10.717157  13.084694  4.800246
4   chronos_price      JPM  10.855528  12.101921  3.467692
5   chronos_price      JNJ   9.587438  11.170395  4.211882
6   chronos_price      WMT   3.756565   4.487598  3.093934
7   chronos_price      SPY   5.607280   6.707737  0.816415
8   chronos_price      XOM   7.582960   8.854527  5.517165
9   chronos_price     NVDA   4.281290   5.235963  2.327393
10  chronos_price  overall   9.564310  13.022071  3.693424
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_chronos_pool_price.parquet
